# 20 — Feature correlation

Within-target Pearson r heatmap across the MD feature block. Kinematic features co-vary strongly. IFP-Tanimoto (interaction-fingerprint similarity) and salt bridges are independent. Tells the Act 4 ML notebooks which features are effectively redundant.

**Method.** Pearson r, computed within each target. Multi-target pooling uses Fisher-z averaging; a naive column-mean of r is NOT reported.

See `docs/GLOSSARY.md` for term definitions.

_(Notebook auto-generated by `reproduce/split_monolith.py`. Self-contained: loads its data via `discovery9.io`, exports figures to `figures/20_feature_correlation_figK.png`.)_


> **Reader guide.** *Experiment A3:* within-target Pearson-r correlation heatmap over the
> per-complex feature bundle, one block per target.
>
> **Method:** Pearson-r on features across ligands per target.
>
> **Reproducibility contract:** reads `data/derived/features.parquet`; r-matrix persisted to
> `data/derived/36_feature_correlation_matrix.csv`.

In [ ]:
# --- notebook preamble ---
NB_STEM = "36_feature_correlation"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')

# --- iter-2 FIX 4: define STABILITY_COLS locally (was lost in monolith split) ---
STABILITY_COLS = ['lig_drift_mean_A', 'lig_drift_std_A', 'lig_com_disp_max_A',
                  'lig_internal_rmsd_mean_A', 'lig_buried_sasa_mean_A2',
                  'lig_buried_sasa_std_A2', 'vdw_contacts_mean', 'n_hb_mean',
                  'rmsd_as_bb_mean_A', 'protein_rg_mean_A']


## 8. Feature correlation matrix — where signal is redundant

Pearson r computed **within each target and averaged across targets**. The correlation reflects joint variation inside a pocket, not cross-target scale differences.

Redundant blocks (|r| > 0.7) can be dropped without losing information. The independent features are the ones worth ensembling.


In [ ]:

corrs = []
for t in df.target.unique():
    sub = df[df.target == t][STABILITY_COLS]
    if len(sub) >= 5:
        corrs.append(sub.corr())
corr = pd.concat(corrs).groupby(level=0).mean().loc[STABILITY_COLS, STABILITY_COLS]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(STABILITY_COLS))); ax.set_xticklabels(STABILITY_COLS, rotation=90, fontsize=8)
ax.set_yticks(range(len(STABILITY_COLS))); ax.set_yticklabels(STABILITY_COLS, fontsize=8)
for i in range(len(STABILITY_COLS)):
    for j in range(len(STABILITY_COLS)):
        v = corr.values[i, j]
        col = WHITE if abs(v) > 0.55 else NAVY
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6, color=col)
cbar = plt.colorbar(im, ax=ax, label='Pearson r (within-target avg)', shrink=0.7)
cbar.ax.yaxis.label.set_color(NAVY)
ax.set_title('Feature correlation — within-target Pearson r, averaged across targets')

**Typical redundant blocks in the MD-stability panel.**
- `lig_drift_*` / `lig_com_disp_*` / `lig_escape_frac` / `lig_rmsf_*` — all measure the same thing: the ligand moved.
- `vdw_contacts_mean` / `lig_buried_sasa_mean_A2` — both measure pocket engagement volume.
- `hb_persistence_frac` / `n_hb_mean` — persistent HB count vs mean.

The small-|r| leftovers are the independent signal. Keep those in the ensemble. `ifp_tanimoto_median_vs_ref` and `salt_bridges_lp_mean` tend to be the most independent of the kinematic block.


In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
